# Corpus filtering

Which files enter the study, and their labels. Writes `data/interim/01_corpus.csv`.

- **Filters:** one section each, the evidence, then the cut.
- **Labels:** final and tonal type, key, transposition.
- **After the labels:** copies (the copy with the better key stays), then notes off the grid.

In [ ]:
import os, json, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import numpy as np
import pandas as pd
from music21 import converter, key, roman
from music21.analysis import discrete

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")   # written by one notebook, read by the next
WORKERS = os.cpu_count() // 2
BEAT = {"bach": 1, "palestrina": 2}   # the beat in quarter notes, as in 00_understanding

pieces = pd.read_csv(INTERIM / "00_pieces.csv").set_index("file")   # one row per file, from 00_understanding
assert pieces.index.is_unique   # no file name occurs in both corpora
is_bach, is_palestrina = pieces["corpus"] == "bach", pieces["corpus"] == "palestrina"
kept = pd.Series(True, index=pieces.index)   # the files still in: each filter narrows it
steps = {"dump of music21": kept.copy()}

def paths(files):   # where each file is in the raw dump
    return [RAW / pieces.at[f, "corpus"] / f for f in files]

def in_parallel(function, args, chunksize=8):   # one call per argument, results in the same order
    with ProcessPoolExecutor(max_workers=WORKERS) as pool:
        return list(pool.map(function, args, chunksize=chunksize))

pieces.groupby("corpus").size().rename("files").to_frame()

## Meter

- **Rule:** a bar of four beats from start to end, 4/4 in Bach and 4/2 in Palestrina. Any other time signature, a change of meter included, removes the file.

In [ ]:
METER = {"bach": "4/4", "palestrina": "4/2"}   # one bar of four beats per corpus
meter_ok = pieces["time_signature"] == pieces["corpus"].map(METER)   # the column lists every meter in the file
display(pieces.assign(kept=meter_ok).groupby(["corpus", "time_signature"]).agg(files=("kept", "size"), kept=("kept", "first")))
kept &= meter_ok
steps["meter"] = kept.copy()
for name, stays in kept.groupby(pieces["corpus"]):
    print(f"{name.capitalize()}: {stays.sum():,} of {len(stays):,} files stay, {(~stays).sum()} leave ({100 * (~stays).mean():.1f}%).")

**Result:** what leaves is mostly triple meter (3/4 in Bach, 3/2 in Palestrina); it stays in the raw data for a later out-of-domain check.

In [ ]:
def record(file, tag):   # a reference record of a kern file
    return next(line.split(":", 1)[1].strip() for line in open(RAW / "palestrina" / file, encoding="latin-1") if line.startswith(f"!!!{tag}:"))

movements = pieces.index[is_palestrina]
title = pd.Series([record(f, "OTL") for f in movements], index=movements).str.replace(r"^[A-Z]\.", "", regex=True)   # the section's opening words, as in "B.Hosanna"
mass = pd.Series([record(f, "OPR") for f in movements], index=movements)
left = ~meter_ok[movements]
hosanna = title.str.contains("osanna", case=False)
print(f"Hosanna: {left[hosanna].sum()} of {hosanna.sum()} leave ({100 * left[hosanna].mean():.1f}%); "
      f"all movements: {left.sum()} of {len(left):,} ({100 * left.mean():.1f}%).")
print(f"Masses that lose at least one movement: {mass[left].nunique()} of {mass.nunique()}.")
title[left].value_counts().head(8).rename("movements that leave").to_frame()

**Result: which Palestrina movements leave**
- Hosanna: 46 of 111 (41.4%), against 8.7% of all movements.
- Then parts of the Credo, *Et resurrexit* (15) and *Et in Spiritum* (10), and codas (12).
- 63 of the 101 masses lose at least one movement.

## Key signatures

- **Check:** one key signature per file, for the transposition; no part may change it. Nothing is cut.

In [ ]:
def key_signatures_of(path):   # per part, its key signatures in order (sharps > 0, flats < 0)
    return [[k.sharps for k in part.recurse().getElementsByClass(key.KeySignature)] for part in converter.parse(path).parts]

signatures = pd.Series(in_parallel(key_signatures_of, paths(pieces.index[kept])), index=pieces.index[kept])
assert not any(len(set(part)) > 1 for parts in signatures for part in parts)   # no part changes its key signature
signatures[[len({part[0] for part in parts if part}) > 1 for parts in signatures]].rename("key signatures, part by part").to_frame()

**Result**
- No part changes its key signature.
- In three Bach chorales the soprano starts in another signature (table); the notes are right, so it is an encoding error, and 00_understanding already takes the signature most parts share.

## Not a chorale

- BWV 846, the Prelude in C major of the Well-Tempered Clavier: a keyboard piece on two staves, the only Bach file with fewer than four parts. It leaves.

In [ ]:
few_parts = kept & is_bach & (pieces["parts"] < 4)
display(pieces.loc[few_parts, ["parts", "measures", "time_signature"]])
kept &= ~few_parts
steps["not a chorale"] = kept.copy()

## Voices

- **Bach:** the four sung voices; instrumental parts leave, and so does a chorale with more than four voices.
- **Palestrina:** every voice stays, because a mass mixes movements with different numbers of voices (00_understanding).
- **Line:** the line written in its part, also where two voices cross.
- **Type:** from the instrument code (`*Icant` Soprano, `*Icalto` Alto, then Tenor, Baritone, Bass). A repeated type is numbered (Tenor 1, Tenor 2): one type for the counts, two lines in the corpus.

In [ ]:
TYPES = {"bach": ("Soprano", "Alto", "Tenor", "Bass"), "palestrina": ("Soprano", "Alto", "Tenor", "Baritone", "Bass")}

def voices_of(score, corpus):   # the sung parts: all of Palestrina's; Bach's named ones, or the four unnamed parts of a kern file
    if corpus == "palestrina":
        return list(score.parts)
    named = [p for p in score.parts if (p.partName or "").startswith(TYPES["bach"])]
    return named or (list(score.parts) if all(p.partName is None for p in score.parts) else [])

def lines_of(path):   # which parts are voices, the label of each line, and how many notes the voices sing
    score = converter.parse(path)
    voices = voices_of(score, path.parent.name)
    if all(p.partName is None for p in voices):   # kern spines, listed from the top voice down
        means = [np.mean([n.pitches[0].midi for n in p.flatten().notes]) for p in voices]
        assert len(voices) == 4 and all(a > b for a, b in zip(means, means[1:])), path.name
        types = list(TYPES["bach"])
    else:
        types = [p.partName.split()[0] for p in voices]   # "Soprano 1" -> "Soprano"
        assert set(types) <= set(TYPES[path.parent.name]), (path.name, types)
    notes = [len(p.recurse().notes) for p in voices]
    assert min(notes) > 0, path.name   # no voice is silent for a whole movement
    return {"voices": len(voices), "voice_parts": "|".join(str(list(score.parts).index(p)) for p in voices),
            "lines": "|".join(f"{t} {types[:k + 1].count(t)}" if types.count(t) > 1 else t for k, t in enumerate(types)),
            "voice_notes": sum(notes)}

lines = pd.DataFrame(in_parallel(lines_of, paths(pieces.index[kept])), index=pieces.index[kept])
pieces = pieces.drop(columns=lines.columns, errors="ignore").join(lines)
kept &= is_palestrina | (pieces["voices"] == 4)
steps["voices"] = kept.copy()

def with_voices(files):   # (path, indices of the voice parts) for each file
    return [(path, [int(k) for k in pieces.at[path.name, "voice_parts"].split("|")]) for path in paths(files)]

def sung(path, parts):   # a score with its voice parts only
    score = converter.parse(path)
    for k, part in enumerate(list(score.parts)):
        if k not in parts:
            score.remove(part)
    return score

In [ ]:
looked = pieces[steps["not a chorale"]].astype({"voices": int})   # the files this section read
bach_looked, palestrina_looked = looked[looked["corpus"] == "bach"], looked[looked["corpus"] == "palestrina"]
more = bach_looked[bach_looked["voices"] != 4]
repeated = palestrina_looked["lines"].str.contains(" [0-9]", regex=True)
print(f"Bach: {(bach_looked['parts'] > bach_looked['voices']).sum()} chorales lose their instrumental parts; leaving, with more than four voices: "
      + ", ".join(f"{f} ({lines})" for f, lines in more["lines"].items()) + ".")
print(f"Palestrina: {len(palestrina_looked):,} movements; {(palestrina_looked['lines'] == 'Soprano|Alto|Tenor|Bass').sum()} are exactly "
      f"Soprano, Alto, Tenor, Bass; {repeated.sum()} number a repeated type.")
display(looked.groupby(["corpus", "voices"]).size().unstack(fill_value=0))
palestrina_looked["lines"].value_counts().head(8).rename("movements").to_frame()

**Result**
- Bach: 32 chorales lose their instrumental parts; BWV 227.3 (two sopranos) leaves; every chorale left has four voices.
- Palestrina: of 1,203 movements, 290 are exactly Soprano, Alto, Tenor, Bass; 834 number a repeated type, most often two tenors.

## Bars

- **Rule:** every bar holds four beats.
- **Case:** Bach's pickups, and bars split in two at a repeat or a phrase end; music21 pads the missing beats, so beats keep their numbers.
- **Check, on the top voice:** every whole bar starts a whole number of bars after the first. A piece without its partner, or a bar of another length, breaks the pattern and removes the file; stray pieces before the first or after the last whole bar are counted too.

In [ ]:
def bars_of(args):   # pickup, split bars, whole bars off the four-beat pattern, and stray pieces the check cannot see
    path, part = args
    bar = 4 * BEAT[path.parent.name]
    measures = list(converter.parse(path).parts[part].getElementsByClass("Measure"))
    lengths = [float(m.duration.quarterLength) for m in measures]
    whole = [k for k, length in enumerate(lengths) if np.isclose(length, bar)]
    phases = [(float(measures[k].offset) - float(measures[whole[0]].offset)) % bar for k in whole]
    return {"pickup": lengths[0] < bar and not np.isclose(lengths[0], bar),
            "split_bars": sum(not np.isclose(length, bar) for length in lengths[1:-1]),
            "off_pattern": sum(not (np.isclose(x, 0) or np.isclose(x, bar)) for x in phases),
            "unchecked": sum(not np.isclose(lengths[k], bar) for k in [*range(1, whole[0]), *range(whole[-1] + 1, len(lengths) - 1)]),
            "irregular": ", ".join(f"bar {m.number}: {length:g}" for m, length in zip(measures[1:-1], lengths[1:-1]) if not np.isclose(length, bar))}

files = pieces.index[kept]
bars = pd.DataFrame(in_parallel(bars_of, [(path, parts[0]) for path, parts in with_voices(files)]), index=files)
display(bars.groupby(pieces.loc[files, "corpus"]).agg(files=("pickup", "size"), pickup=("pickup", "sum"), split_bars=("split_bars", lambda s: (s > 0).sum()),
                                                      off_pattern=("off_pattern", lambda s: (s > 0).sum()), unchecked=("unchecked", "sum")))
kept &= ~pieces.index.isin(bars.index[bars["off_pattern"] > 0])
steps["bars"] = kept.copy()
bars.loc[bars["off_pattern"] > 0, ["irregular", "off_pattern"]]

**Result**
- Bach: 266 chorales start with a pickup, 206 split a bar.
- 6 leave: a bar of ten beats (`bwv324`), a bar of two (`bwv261`), pieces without their partner (`bwv48.3`, `bwv374`, `bwv276`, `bwv4.8`).
- No stray pieces. Palestrina has neither pickups nor split bars.

## Final and tonal type

Labels of the files that remain.

- **Final:** the lowest note of the closing sonority.
- **Tonal type** ([Powers 1981](https://doi.org/10.2307/831189)): system (*cantus durus*, no flat; *cantus mollis*, one flat), cleffing (*chiavi naturali* or *chiavette*), final.
- **Palestrina:** system and final, since the files carry modern clefs. No mode is named: a tonal type is not a mode (Powers 1981).
- **Bach:** the final only says whether a chorale closes on its tonic.

In [ ]:
NAME12 = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]

def final_of(args):   # pitch class of the lowest note in the closing sonority of the voices
    lasts = [part.flatten().notes[-1] for part in sung(*args).parts]
    end = max(float(n.offset + n.quarterLength) for n in lasts)
    return min(p.midi for n in lasts if abs(float(n.offset + n.quarterLength) - end) < 1e-6 for p in n.pitches) % 12

files = pieces.index[kept]
pieces["final"] = pd.Series([NAME12[f] for f in in_parallel(final_of, with_voices(files))], index=files)
movements = pieces.index[kept & is_palestrina]
assert set(pieces.loc[movements, "key_signature"]) == {0, -1}
system = pieces.loc[movements, "key_signature"].map({0: "durus", -1: "mollis"})
pieces["tonal_type"] = system + ", " + pieces.loc[movements, "final"]
pd.crosstab(pieces.loc[movements, "final"], system, margins=True)

**Result**
- 1,203 movements: 748 in *cantus durus*, 455 in *cantus mollis*; the last two filters remove 24.
- Commonest: durus G (262), mollis G (191), mollis F (156), durus D (147), durus A (144). Rare: durus F (6), mollis E (3), mollis A (1).

## Key of the Bach chorales

- **Why experts:** chorales modulate and some close on the dominant, so the key is an interpretation.
- **Source:** When in Rome ([Gotham et al. 2023](https://doi.org/10.5334/tismir.165), CC BY-SA 4.0): Roman-numeral analyses of the Riemenschneider chorales, by Tymoczko and collaborators, linked to the music21 files.
- **Rule:** the first key of the analysis. An analysis in another key moves by the interval between first chords, checked on the last chords.

In [ ]:
REPO, COMMIT = "MarkGotham/When-in-Rome", "1c61fe41b8c2910296d7d2bcbf6476c7c1f2fe35"   # 2026-05-27
CHORALES = "Corpus/Early_Choral/Bach,_Johann_Sebastian/Chorales"   # one folder per Riemenschneider number
ANALYSES = RAW / "when_in_rome"
listing = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/git/trees/{COMMIT}:{CHORALES}?recursive=1"))
assert not listing["truncated"]
blobs = [e["path"] for e in listing["tree"] if e["type"] == "blob"]
missing = [f for f in blobs if not (ANALYSES / f).exists()]
for f in tqdm(missing, desc="when_in_rome"):
    (ANALYSES / f).parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{urllib.request.quote(CHORALES)}/{f}", ANALYSES / f)
print(f"when_in_rome: {len(blobs)} files in {len({f.split('/')[0] for f in blobs})} chorale folders, {len(missing)} downloaded")

In [ ]:
def numerals_of(path):   # tonic and mode of the first key, roots of the first and last numeral
    numerals = list(converter.parse(path, format="romantext").recurse().getElementsByClass(roman.RomanNumeral))
    return numerals[0].key.tonic.pitchClass, numerals[0].key.mode, numerals[0].root().pitchClass, numerals[-1].root().pitchClass

def ends_of(args):   # roots of the first and last sonority of the voices
    chords = sung(*args).chordify().flatten().notes
    return chords[0].root().pitchClass, chords[-1].root().pitchClass

def key_in_score(numerals, ends):   # the analysis' key moved into the score's key; None if first and last chord disagree on the move
    tonic, mode, first, last = numerals
    move = {(ends[0] - first) % 12, (ends[1] - last) % 12}
    return f"{NAME12[(tonic + move.pop()) % 12]} {mode}" if len(move) == 1 else None

matches = {}   # music21 file -> lowest Riemenschneider number that analyses it
for folder in sorted(ANALYSES.iterdir()):
    for name in json.load(open(folder / "remote.json")).get("music21") or []:
        matches.setdefault(name, folder.name)
chorales = pieces.index[kept & is_bach]
analysed = [f for f in chorales if f in matches]
numerals = dict(zip(analysed, in_parallel(numerals_of, [ANALYSES / matches[f] / "analysis.txt" for f in analysed])))
ends = dict(zip(analysed, in_parallel(ends_of, with_voices(analysed))))
expert = pd.Series({f: key_in_score(numerals[f], ends[f]) for f in analysed}).dropna()   # the experts' keys, in the score's key
moved = sum((ends[f][0] - numerals[f][2]) % 12 != 0 for f in expert.index)
print(f"Bach chorales: {len(chorales)}; with an analysis: {len(analysed)}; aligned with the score: {len(expert)}, {moved} of them moved from another key; "
      f"not aligned: {', '.join(sorted(set(analysed) - set(expert.index)))}; without an analysis: {len(chorales) - len(analysed)}.")
expert.str.split().str[1].value_counts().rename("chorales").to_frame()

**Result**
- 354 chorales: 306 analysed, 304 aligned with their score (15 moved from another key); 154 minor, 150 major.
- 50 without an expert key: 48 not analysed; in `bwv117.4` and `bwv227.7` first and last chords disagree, most likely different versions.

### How reliable the expert key is

- **Second expert:** BCMH ([Sears et al. 2023](https://doi.org/10.1037/xhp0001103)), 100 of the first 101 chorales.
- **Without an expert:** Krumhansl–Schmuckler ([Krumhansl 1990](https://doi.org/10.1093/acprof:oso/9780195148367.001.0001)) with music21's five key profiles; the key signature (a major key or its relative minor); the final as tonic.
- **Scores:** same key, and same shift (section Transposition); a relative key gives the same shift, last column.

In [ ]:
PROFILES = {"Krumhansl–Kessler": discrete.KrumhanslSchmuckler, "Aarden–Essen (music21's default)": discrete.AardenEssen,
            "Bellman–Budge": discrete.BellmanBudge, "Temperley–Kostka–Payne": discrete.TemperleyKostkaPayne,
            "Simple weights": discrete.SimpleWeights}

def profiles_of(args):   # one key per profile, from the notes of the whole chorale
    score = sung(*args)
    return [f"{NAME12[k.tonic.pitchClass]} {k.mode}" for k in (cls().getSolution(score) for cls in PROFILES.values())]

def nearest(semitones):   # the same move, up or down, whichever is shorter
    s = semitones % 12
    return s - 12 if s > 6 else s

def shift_of(key_name):   # semitones that bring the tonic to C (major) or A (minor)
    name, mode = key_name.split()
    return nearest((0 if mode == "major" else 9) - NAME12.index(name))

found = pd.DataFrame(in_parallel(profiles_of, with_voices(chorales), chunksize=4), index=chorales, columns=list(PROFILES))
second_files = [f for f in expert.index if (ANALYSES / matches[f] / "analysis_BCMH.txt").exists()]
second = pd.Series([key_in_score(n, ends[f]) for f, n in zip(second_files, in_parallel(numerals_of, [ANALYSES / matches[f] / "analysis_BCMH.txt" for f in second_files]))],
                   index=second_files).dropna()
signature_shift = pieces.loc[chorales, "key_signature"].map(lambda s: nearest(-7 * s))   # the shift of both keys a signature names
truth = expert.map(shift_of)

def agreement(same_key, same_shift=None):   # how often a method agrees with the experts
    row = {"of": len(same_key), "same key %": round(100 * same_key.mean(), 1), "same shift %": np.nan, "wrong keys that are the relative %": np.nan}
    if same_shift is not None:
        row["same shift %"] = round(100 * same_shift.mean(), 1)
        row["wrong keys that are the relative %"] = round(100 * (same_shift & ~same_key).sum() / (~same_key).sum(), 1)
    return row

named = pd.Series([k in {f"{NAME12[7 * s % 12]} major", f"{NAME12[(7 * s + 9) % 12]} minor"} for k, s in zip(expert, pieces.loc[expert.index, "key_signature"])], index=expert.index)
rows = {"second expert (BCMH)": agreement(second == expert[second.index], second.map(shift_of) == truth[second.index])}
rows |= {f"key finding, {p}": agreement(found.loc[expert.index, p] == expert, found.loc[expert.index, p].map(shift_of) == truth) for p in PROFILES}
rows["key signature (one of its two keys)"] = agreement(named, signature_shift[expert.index] == truth)
rows["final as tonic"] = agreement(pieces.loc[expert.index, "final"] == expert.str.split().str[0])
pd.DataFrame(rows).T.astype({"of": int})

**Result**
- Two experts agree on 97.7% of the 88 chorales they share.
- Key: the best profile, Aarden–Essen (music21's default), reaches 88.8%, and music21 warns that the source of its minor-key weights is unknown; Krumhansl–Kessler, the original, 75.0%.
- Shift: four profiles reach 94.4% to 96.4% (half or more of their errors are the relative key); Krumhansl–Kessler 78.9%; the key signature 87.2%; the final as tonic 85.5%.
- So the key is the experts' wherever they analysed the chorale.

### Chorales without an expert key

Two kinds of evidence, the notes (five profiles) and the notation (key signature):

1. At least three profiles give one shift, and the signature gives it too: that shift ("majority of profiles with the signature").
2. Otherwise all five profiles agree: their shift, against the signature ("all five profiles"). Bach sometimes writes one accidental fewer in the signature and the missing one in the score.

- **Key:** the one most of those profiles name.
- **Why a majority:** the profiles err in known directions. Humdrum's `keycor` documentation, quoted by music21: Krumhansl–Kessler tends strongly to read the dominant as tonic, Aarden–Essen weakly the subdominant.
- **Caveat:** no parameter, but chosen among three rules on the chorales with an expert key, so its score there is slightly optimistic.
- `key_source` records where each key comes from.

In [ ]:
shifts = found.map(shift_of)
tally = pd.DataFrame([Counter(row).most_common(1)[0] for row in shifts.itertuples(index=False)], index=shifts.index, columns=["shift", "votes"])
step1 = (tally["votes"] >= 3) & (tally["shift"] == signature_shift)   # majority of profiles with the signature
step2 = ~step1 & (tally["votes"] == 5)                                # all five profiles, against the signature
rules = {"majority of profiles with the signature, alone": step1, "all five profiles, alone": tally["votes"] == 5, "the rule: step 1, else step 2": step1 | step2}
display(pd.DataFrame({name: {"covered": int(mask[truth.index].sum()), "of": len(truth),
                             "same shift %": round(100 * (tally["shift"][truth.index] == truth)[mask[truth.index]].mean(), 1)} for name, mask in rules.items()}).T.astype({"covered": int, "of": int}))

todo = chorales.difference(expert.index)
assert len(expert) + len(todo) == len(chorales) and (step1 | step2)[todo].all()   # every chorale gets a key, from the experts or from the rule
by_rule = pd.Series({f: Counter(k for k, s in zip(found.loc[f], shifts.loc[f]) if s == tally.at[f, "shift"]).most_common(1)[0][0] for f in todo})
pieces["key"] = pd.concat([expert, by_rule])
pieces["key_source"] = pd.concat([pd.Series("expert (When in Rome)", index=expert.index),
                                  pd.Series(np.where(step1[todo], "majority of profiles with the signature", "all five profiles"), index=todo)])
pieces.loc[chorales, "key_source"].value_counts().rename("chorales").to_frame()

**Result**, on the 304 chorales with an expert key
- Step 1 alone: right shift for 98.4% of the 257 it covers; all five profiles: 97.5% of 243.
- The rule covers 285 and gives the right shift for 97.2%, against 97.7% between two experts.
- It covers all 50 chorales without an expert key: 47 by step 1, 3 by step 2.

## Transposition

- **Rule:** to the white keys. Bach: tonic to C if major, to A if minor, as BachBot ([Liang et al. 2017](https://archives.ismir.net/ismir2017/paper/000156.pdf)). Palestrina: *cantus mollis* down a fourth into *cantus durus*.
- **Shift:** the nearest, at most six semitones, so voices keep their range.
- **Stored, not applied:** the grid notebook applies it; a generated piece takes back the key of its melody.

In [ ]:
pieces["shift"] = pd.concat([pieces.loc[kept & is_bach, "key"].map(shift_of),
                             pieces.loc[kept & is_palestrina, "key_signature"].map({0: 0, -1: -5})]).astype("Int64")
pieces[kept].groupby("corpus")["shift"].value_counts().sort_index().rename("files").to_frame().T

**Result:** Bach shifts from five semitones down to five up; in Palestrina the 455 movements in *cantus mollis* move down five, the 748 in *cantus durus* stay.

## Copies

- **Why:** the Riemenschneider edition prints some chorales twice ([Dahn 2018](http://bach-chorales.com/ChoraleDuplicates.htm)), and music21 adds versions.
- **Candidates:** pairs that share at least a fifth of their six-interval fragments, in any voice; transposed copies included.
- **Measure:** voice by voice, pitches relative to the top voice's last note: the share of the shorter file's notes missing from the longer.
- **Identical:** enter once; the copy with an expert key stays, then the MusicXML file.
- **Versions, up to 7.4%:** all stay, same `copy_group`, weight 1/k in 02_corpus.
- **Different harmonisations:** enter whole.
- **Written-out repeat:** a version of the file with a repeat sign; 02_corpus drops the written-out repeat.

In [ ]:
NEAR = 7.4   # % of differing notes up to which two files are versions of one piece (set by the gap in the table below)

def lines_of(args):   # per voice, (onset, duration, pitch) of each note, tied notes merged; the top voice first
    path, parts = args
    score = converter.parse(path)
    return [[(float(n.offset), float(n.quarterLength), max(p.midi for p in n.pitches)) for n in score.parts[k].stripTies().flatten().notes] for k in parts]

def matched(a, b):   # notes of a found in b, in order
    return sum(block.size for block in SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks())

files = pieces.index[kept]
lines = dict(zip(files, in_parallel(lines_of, with_voices(files))))
fragments = {f: {tuple(s[i:i + 6]) for s in ([b[2] - a[2] for a, b in zip(line, line[1:])] for line in voices) for i in range(len(s) - 5)}
             for f, voices in lines.items()}
holders = defaultdict(set)
for f, s in fragments.items():
    for x in s:
        holders[x].add(f)
shared = defaultdict(int)
for x, holding in holders.items():
    if len(holding) <= 50:   # a fragment in more than 50 files is a formula, not a sign of a copy
        holding = sorted(holding)
        for i, a in enumerate(holding):
            for b in holding[i + 1:]:
                shared[a, b] += 1
candidates = [(a, b) for (a, b), n in shared.items()
              if n >= 0.2 * min(len(fragments[a]), len(fragments[b])) and pieces.at[a, "corpus"] == pieces.at[b, "corpus"] and pieces.at[a, "voices"] == pieces.at[b, "voices"]]

def compare(a, b):   # identical, and the share of the shorter file's notes the longer one lacks
    rel = lambda voices: [[(p - voices[0][-1][2], d) for _, d, p in line] for line in voices]
    A, B = rel(lines[a]), rel(lines[b])
    short = sum(min(len(x), len(y)) for x, y in zip(A, B))
    lack = short - sum(matched([p for p, _ in x], [p for p, _ in y]) for x, y in zip(A, B))
    return A == B, round(100 * lack / short, 1)

copies = pd.DataFrame([(a, b, *compare(a, b)) for a, b in candidates], columns=["a", "b", "identical", "differ %"])
copies["corpus"] = copies["a"].map(pieces["corpus"])
copies["kind"] = np.select([copies["identical"], copies["differ %"] <= NEAR], ["identical", "versions"], "different")
display(copies.groupby(["corpus", "kind"]).size().unstack(fill_value=0))
copies.sort_values("differ %").groupby("corpus").head(26)[["corpus", "a", "b", "differ %", "kind"]]

In [ ]:
# checks: every duplicate of Dahn (2018) with both files in the corpus is found, and Palestrina's identical files are one movement encoded twice
DAHN = [("bwv12.7.mxl", "bwv69.6-a.mxl"), ("bwv175.7.mxl", "bwv59.3.mxl"), ("bwv244.15.mxl", "bwv244.17.mxl"),
        ("bwv253.mxl", "bwv414.mxl"), ("bwv197.7-a.mxl", "bwv398.mxl"), ("bwv245.5.mxl", "bwv416.mxl")]
same_piece = copies[copies["kind"] != "different"]
found = {frozenset(p) for p in zip(same_piece["a"], same_piece["b"])}
assert all(frozenset(p) in found for p in DAHN if set(p) <= set(files))
twins = copies[(copies["corpus"] == "palestrina") & (copies["kind"] == "identical")]
assert all(record(a, "OPR") == record(b, "OPR") and record(a, "YOR") == record(b, "YOR") for a, b in zip(twins["a"], twins["b"]))
print(f"Dahn's duplicates in the corpus: {sum(set(p) <= set(files) for p in DAHN)}, all found. "
      f"Palestrina: {len(twins)} identical pairs, each with the same mass and the same page of the edition.")

parent = {f: f for f in files}   # copies joined into one piece
def root(f):
    while parent[f] != f:
        f = parent[f]
    return f
for a, b in zip(same_piece["a"], same_piece["b"]):
    parent[root(a)] = root(b)
def preference(f):   # the copy that stays among identical ones
    return (pieces.at[f, "key_source"] != "expert (When in Rome)", not f.endswith(".mxl"), f)
dropped = {sorted(pair, key=preference)[1] for pair in zip(copies.loc[copies["kind"] == "identical", "a"], copies.loc[copies["kind"] == "identical", "b"])}
kept &= ~pieces.index.isin(list(dropped))
steps["copies"] = kept.copy()
roots = pd.Series({f: root(f) for f in files})[pieces.index[kept]]
pieces["copy_group"] = roots.map({r: k for k, r in enumerate(sorted(set(roots)))})
print(f"Identical copies left out: {len(dropped)} ({', '.join(sorted(dropped))}).")
print(f"Pieces with more than one version: {(pieces.loc[kept, 'copy_group'].value_counts() > 1).sum()}, in {pieces.loc[kept].groupby('copy_group').filter(lambda g: len(g) > 1).shape[0]} files.")

**Result**
- Bach: 2 identical pairs, 19 pairs of versions (0% to 7.4%). The next pair differs by 11.1% (`bwv10.7`, `bwv323`, another harmonisation of a shared tune), so 7.4% sits in a gap.
- Palestrina: 21 identical pairs, each one movement encoded twice (same mass, same page of the edition); one pair of versions, `Gloria_18` and `Gloria_of_1600` (2.8%); the next pair 10.7%.
- 23 identical copies leave; 20 pieces keep more than one version, 40 files.

## Notes off the grid

- **Grid:** a quarter of the beat (a sixteenth for Bach, an eighth for Palestrina); one note per voice per cell.
- **Problem:** a note that starts between grid lines gives way to the note of its cell, or its file leaves.
- **Scope:** the voices of the files still in (00_understanding counted every part).

In [ ]:
def off_grid_notes(path):   # every note in a voice that starts between two grid lines, with the notes around it
    corpus = path.parent.name
    step = BEAT[corpus] / 4
    rows = []
    for part in voices_of(converter.parse(path), corpus):
        notes = list(part.flatten().notes)
        for k, n in enumerate(notes):
            if not np.isclose(float(n.offset) / step, round(float(n.offset) / step)):
                rows.append({"corpus": corpus, "file": path.name, "part": part.partName, "bar": n.measureNumber, "duration": n.duration.type,
                             "tuplet": "/".join(f"{t.numberNotesActual}:{t.numberNotesNormal}" for t in n.duration.tuplets) or "-",
                             "before": " ".join(f"{x.nameWithOctave}({float(x.quarterLength):g})" for x in notes[max(k - 2, 0):k]),
                             "note": f"{n.nameWithOctave}({float(n.quarterLength):g})", "after": notes[k + 1].nameWithOctave if k + 1 < len(notes) else None})
    return rows

off = pd.DataFrame([row for rows in in_parallel(off_grid_notes, paths(pieces.index[kept])) for row in rows])
share = off.groupby("corpus").agg(notes=("file", "size"), files=("file", "nunique"))
share["% of the voice notes"] = (100 * share["notes"] / pieces[kept].groupby("corpus")["voice_notes"].sum()).round(3)
display(share)
display(off.groupby(["corpus", "duration", "tuplet"]).agg(notes=("file", "size"), files=("file", "nunique")))
off[off["corpus"] == "bach"].drop(columns=["corpus", "duration", "tuplet"])

**Result**
- Bach: always the second of two thirty-seconds after a dotted eighth in the soprano, a written-out ornament.
- Palestrina: triplet minims, plus one whole note after a triplet passage.
- Copies such as `bwv248.64-6` and `bwv248.64-s` are handled in Copies.

### Bach: each cell keeps the note sounding at its end

Two thirty-seconds share one sixteenth; the cell keeps one.

- **DeepBach** ([Hadjeres et al. 2017](https://proceedings.mlr.press/v70/hadjeres17a.html); code: [`part_to_tensor`](https://github.com/Ghadjeres/DeepBach/blob/master/DatasetManager/chorale_dataset.py)): the note at the start of the cell, the first.
- **BachBot** ([Liang et al. 2017](https://archives.ismir.net/ismir2017/paper/000156.pdf); code: [`encode_score`](https://github.com/feynmanliang/bachbot/blob/master/scripts/datasets.py)): music21's `quantize` to the nearest line; here the second lies exactly halfway.
- **Ours:** the note at the end of the cell, the second; notes on the grid do not change.
- **Check:** the kept note must belong to the chord (every note sounding when the dotted eighth starts, itself included) and the dropped one not; the next cell stops otherwise. A stricter chord, the other voices only, is printed too.

In [ ]:
SIXTEENTH = BEAT["bach"] / 4

def cells(part, step, at_end):   # one pitch per grid cell: the note sounding at the start of the cell, or at its end
    notes = [(float(n.offset), float(n.offset + n.quarterLength), n.pitches[-1].midi) for n in part.flatten().notes]
    probe = step - 1e-6 if at_end else 0.0
    return [next((p for start, end, p in notes if start <= t + probe < end), None) for t in np.arange(0, notes[-1][1], step)]

def ornaments(path):   # each off-grid note: the note the end rule keeps, the one it drops, and whether each belongs to the chord
    parts = voices_of(converter.parse(path), "bach")
    lines = [list(p.flatten().notes) for p in parts]
    def sounding(at, which): return {p.pitchClass for line in which for x in line if x.offset <= at < x.offset + x.quarterLength for p in x.pitches}
    rows = []
    for v, line in enumerate(lines):
        for k, n in enumerate(line):
            if np.isclose(float(n.offset) / SIXTEENTH, round(float(n.offset) / SIXTEENTH)):
                continue
            first, decorated = line[k - 1], line[k - 2]
            assert n.quarterLength == first.quarterLength == SIXTEENTH / 2 and decorated.quarterLength == 3 * SIXTEENTH, (path.name, n.measureNumber)
            chord, moment = sounding(decorated.offset, lines), sounding(first.offset, lines[:v] + lines[v + 1:])
            rows.append({"file": path.name, "bar": n.measureNumber, "decorated": decorated.name, "dropped": first.name, "kept": n.name,
                         "kept in chord": n.pitch.pitchClass in chord, "dropped in chord": first.pitch.pitchClass in chord,
                         "kept, other voices only": n.pitch.pitchClass in moment, "dropped, other voices only": first.pitch.pitchClass in moment})
    changed = sum(sum(a != b for a, b in zip(cells(p, SIXTEENTH, False), cells(p, SIXTEENTH, True))) for p in parts)
    assert rows and changed == len(rows), (path.name, changed, len(rows))   # the end rule changes the ornament cells and nothing else
    return rows

check = pd.DataFrame([row for rows in in_parallel(ornaments, paths(off.loc[off["corpus"] == "bach", "file"].unique())) for row in rows])
assert len(check) == (off["corpus"] == "bach").sum() and check["kept in chord"].all() and not check["dropped in chord"].any()
print(f"{len(check)} ornaments in {check['file'].nunique()} chorales. The end rule keeps a chord tone in {check['kept in chord'].sum()} and drops one in "
      f"{check['dropped in chord'].sum()}. Counting only the other voices at the moment: kept {check['kept, other voices only'].sum()}, dropped {check['dropped, other voices only'].sum()}.")
check

**Result**
- All nine ornaments: the kept note is in the chord, the dropped one is not.
- Stricter chord: the kept note sounds 6 times out of 9, the dropped one never; the other three (`bwv185.6` twice, `bwv36.4-2`) need the decorated note.
- Only these nine cells change; the grid notebook uses `cells(part, step, at_end=True)`.

### Palestrina: the movements with triplets leave

- **Why:** their triplet minims put three minims in the time of two, a change of meter written with tuplets; the meter filter's rule, one meter of four duple beats throughout, applies.
- **Why not as in Bach:** a triplet minim lasts two thirds of a beat; on an eighth-note grid the second and third notes move (the end rule takes 1⅓ and 2⅔ quarter notes to 1 and 2½), a duple rhythm Palestrina did not write.
- **Why whole movements:** cutting only the passages would leave segments, not pieces.

In [ ]:
triplet_files = off.loc[off["corpus"] == "palestrina", "file"].unique()
assert all((off.loc[off["file"] == f, "tuplet"] == "3:2").any() for f in triplet_files)   # each file has triplet minims
movements = off[off["corpus"] == "palestrina"].groupby("file").size().rename("off-grid notes").to_frame()
movements["voice notes"] = pieces.loc[movements.index, "voice_notes"].astype(int)
movements["%"] = (100 * movements["off-grid notes"] / movements["voice notes"]).round(1)
kept &= ~pieces.index.isin(triplet_files)
steps["notes off the grid"] = kept.copy()
movements

**Result:** off-grid notes are 21.3% of `Agnus_III_b`, 34.4% of `Benedictus_66_b`, 5.4% of `Kyrie_89`; the three leave.

## The filtered corpus

- **Table:** files and notes after each filter; notes of every part until the voices step, of the voices after.
- **Last column:** each step's share of all the notes a corpus loses.

In [ ]:
rows = []
for name, mask in steps.items():
    column = "notes" if name in ("dump of music21", "meter", "not a chorale") else "voice_notes"
    for corpus_name, group in pieces[mask].groupby("corpus"):
        rows.append({"corpus": corpus_name, "after": name, "files": len(group), "notes": int(group[column].sum())})
retention = pd.DataFrame(rows).sort_values("corpus", kind="stable").set_index(["corpus", "after"])
first = retention.groupby("corpus").transform("first")
retention["files %"] = (100 * retention["files"] / first["files"]).round(1)
retention["notes %"] = (100 * retention["notes"] / first["notes"]).round(1)
lost = first["notes"] - retention.groupby("corpus")["notes"].transform("last")
retention["share of the notes lost %"] = (100 * retention.groupby("corpus")["notes"].diff().fillna(0).abs() / lost).round(1)
retention

**Result**
- Bach keeps 352 of 413 files (85.2%) and 73.9% of its notes; of the notes lost, the meter takes 55.1%, the voices 34.7% (instrumental parts, BWV 227.3), the copies 2.4%.
- Palestrina keeps 1,179 of 1,318 movements (89.5%) and 94.6% of its notes; the meter takes 66.3% of the loss, the movements encoded twice 29.3%.

## Against the literature

How the closest works prepare the same Bach chorales, read in their papers and code on 2026-10-23 and 24; a dash where they say nothing.

| Step | DeepBach (Hadjeres et al. 2017) | BachBot (Liang et al. 2017) | BacHMMachine (Zhu et al. 2021) | This notebook |
|---|---|---|---|---|
| Meter | no filter: "Bach chorales only contain simple time signatures" | 4/4 only, read on the first time signature (`iter_standardized_chorales`) | — | 4/4 from start to end; a change of meter takes the file out |
| Instrumental parts | the whole chorale out: "after removing chorales with instrumental parts", and the code keeps only four-part scores (`is_valid`) | the parts named Soprano, Alto, Tenor and Bass stay, the others are removed (`extract_SATB`) | — | the instrumental parts are removed, the 32 chorales stay with their four sung voices |
| Five sung voices (BWV 227.3) | out, with every score that does not have four parts | "Soprano 1" kept as the soprano, "Soprano 2" dropped; the code itself asks "soprano1 or soprano2?" | — | out: reducing it to four would drop a sung line |
| Voice divisions | chorales "containing parts with two simultaneous notes" out | — | — | none in the sung voices of the 1,531 files kept (checked 2026-10-24) |
| Bars | — | — | — | six chorales whose downbeats leave the four-beat pattern out |
| Grid | sixteenth note; the note sounding at the start of each cell | sixteenth note; music21's `quantize` to the nearest line | quarter note, one step per beat | a quarter of the beat; the note sounding at the end of each cell |
| Copies | — | — | — | identical copies enter once; versions weighted 1/k in 02_corpus |
| Key and transposition | no common key: every transposition within the voice ranges is added | to C major or A minor by music21's key finding (Aarden–Essen) | to C major or A minor | expert keys from When in Rome, a fixed rule elsewhere; to the white keys by the nearest shift |

- **Allan and Williams (2004):** separate models for major and minor chorales, one time step per beat.
- **Palestrina:** Anders and Inden (2019) use the first 36 Agnus movements of the music21 corpus, all in 4/2; this notebook keeps every movement in 4/2, with all its voices.


## The corpus table

One row per file that enters the study, with the columns the next notebooks read.

In [ ]:
COLUMNS = ["corpus", "voices", "lines", "voice_parts", "voice_notes", "key_signature", "final", "tonal_type", "key", "key_source", "shift", "copy_group"]
table = pieces.loc[kept, COLUMNS].astype({"voices": int, "voice_notes": int})
table.to_csv(INTERIM / "01_corpus.csv")   # one row per file, read by 02_corpus
table.groupby("corpus").size().rename("files").to_frame()